In [1]:
!pip install fastapi uvicorn nest-asyncio pyngrok -q
!pip install pygltflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00


In [7]:
import os
from google.colab import userdata
os.environ['NGROK_AUTHTOKEN']= userdata.get('NGROK_AUTHTOKEN')

In [8]:
from pygltflib import GLTF2
import os
import shutil

def extract_textures_and_copy_model(glb_path, output_dir, image_path = None):
    os.makedirs(output_dir, exist_ok=True)

    gltf = GLTF2().load(glb_path)

    glb_filename = os.path.basename(glb_path)
    copied_glb_path = os.path.join(output_dir, glb_filename)
    shutil.copy2(glb_path, copied_glb_path)
    print(f"Copied model to {copied_glb_path}")

    if image_path is not None and os.path.exists(image_path):
        image_filename = os.path.basename(image_path)
        copied_image_path = os.path.join(output_dir, image_filename)
        shutil.copy2(image_path, output_dir)
        print(f"Copied image to {copied_image_path}")

    with open(glb_path, 'rb') as f:
        content = f.read()

    def get_bin_chunk():
        magic = int.from_bytes(content[0:4], 'little')
        assert magic == 0x46546C67  # b'glTF'
        json_len = int.from_bytes(content[12:16], 'little')
        json_type = content[16:20]
        assert json_type == b'JSON'

        bin_offset = 20 + json_len
        bin_len = int.from_bytes(content[bin_offset:bin_offset+4], 'little')
        bin_type = content[bin_offset+4:bin_offset+8]
        assert bin_type == b'BIN\x00'

        return content[bin_offset+8 : bin_offset+8+bin_len]

    bin_chunk = get_bin_chunk()
    base_name = os.path.splitext(glb_filename)[0]

    for i, image in enumerate(gltf.images):
        if image.bufferView is None:
            print(f"Skipping image {i} (no bufferView)")
            continue

        buffer_view = gltf.bufferViews[image.bufferView]
        offset = buffer_view.byteOffset or 0
        length = buffer_view.byteLength

        image_data = bin_chunk[offset : offset + length]
        ext = 'png' if image.mimeType == 'image/png' else 'jpg'
        out_path = os.path.join(output_dir, f'{base_name}_texture_{i}.{ext}')

        with open(out_path, 'wb') as f:
            f.write(image_data)
            print(f"Saved texture to {out_path}")


In [9]:
import zipfile
import os

def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                abs_path = os.path.join(root, file)
                rel_path = os.path.relpath(abs_path, folder_path)  # relative path inside zip
                zipf.write(abs_path, arcname=rel_path)
    print(f"Zipped folder to: {zip_path}")

In [10]:
from fastapi import FastAPI, Form
from fastapi.responses import FileResponse, JSONResponse
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import os
from fastapi.responses import PlainTextResponse

nest_asyncio.apply()

app = FastAPI()
PORT = 8000

OUTPUT_DIR = "/content/OutputFolder"
CACHE_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "cache")
ZIP_PATH = "/content/output.zip"

@app.get("/")
def read_root():
    return PlainTextResponse("Welcome! POST a filename to /process-3d-file to get a ZIP with model + textures.")


@app.post("/process-3d-file")
def process_3d_file(concept: str = Form(...)):
    list_paths = os.listdir(OUTPUT_DIR)
    glb_path = [
        path for path in list_paths if concept.lower() in os.path.basename(path).lower()
        and path.endswith(".glb") and
        os.path.basename(path).startswith("paint_mesh")
        ][0]
    glb_path = os.path.join(OUTPUT_DIR, glb_path)
    print(glb_path)
    if not os.path.exists(glb_path):
        return JSONResponse({"error": f"GLB file for '{concept}' not cached yet please generate."}, status_code=404)

    glb_heading = "_".split(os.path.basename(glb_path).split('.')[0])[-1]
    image_path_list = [
        # path for path in list_paths if glb_heading.lower() in os.path.basename(path).lower()
        # and (path.endswith(".png") or path.endswith(".jpg") or path.endswith(".jpeg"))
        ]
    image_path = None if len(image_path_list)==0 else os.path.join(OUTPUT_DIR, image_path_list[0])
    print(image_path)

    if os.path.exists(CACHE_OUTPUT_DIR):
        shutil.rmtree(CACHE_OUTPUT_DIR)
    os.makedirs(CACHE_OUTPUT_DIR, exist_ok=True)

    try:
        extract_textures_and_copy_model(glb_path, CACHE_OUTPUT_DIR, image_path)
    except Exception as e:
        return JSONResponse({"error": f"Failed to extract textures: {str(e)}"}, status_code=500)

    try:
        zip_folder(CACHE_OUTPUT_DIR, ZIP_PATH)
    except Exception as e:
        return JSONResponse({"error": f"Failed to zip folder: {str(e)}"}, status_code=500)

    return FileResponse(
        path=ZIP_PATH,
        media_type="application/zip",
        filename=os.path.basename(ZIP_PATH)
    )


if __name__ == "__main__":
    public_url = ngrok.connect(PORT)
    print(f"🌍 Public URL: {public_url}/process-3d-file")
    uvicorn.run(app, host="0.0.0.0", port=PORT)

🌍 Public URL: NgrokTunnel: "https://700211ed6e51.ngrok-free.app" -> "http://localhost:8000"/process-3d-file


INFO:     Started server process [401]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [401]
